<a href="https://colab.research.google.com/github/meijo-yoshikawa-Lab-student/2026labseminar/blob/colabtest/%E5%9E%82%E7%9B%B4%E7%99%BA%E5%B1%95%E8%AA%B2%E9%A1%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 4. 発展課題：電話会社の解約予測（穴埋め・難しめ）

ここからは、本編より少し難しい発展課題です。
別のデータセットで、垂直連合学習をより広い範囲まで自分で組み立てます。
例題（第2章）の `run_vertical_fl` をよく見ながら、空欄を埋めてください。

**お題：電話会社の顧客データで解約を予測する（Telco Customer Churn）**

電話会社の顧客データを使い、ある顧客が解約するかどうかを予測します。
このデータには、顧客の属性・契約しているサービス・契約期間や料金といった、
さまざまな種類の情報が含まれています。

今回はこれらの情報を **3つのグループ** に分け、本編と同じ垂直連合学習の手順で、
それぞれのグループを持ち寄って解約予測ができるか試します。

- グループ1：顧客の属性（性別・高齢者かどうか・家族構成 など）
- グループ2：契約しているサービス（電話・ネット・各種オプション など）
- グループ3：契約期間と料金（契約期間・支払い方法・月額料金 など）

### 4-1. データの準備（穴埋め①〜②）

このデータはインターネット上の公開データを使います（認証は不要です）。
まず、どんなデータなのかを確認しましょう。

In [ ]:
import urllib.request, io

# 公開データを読み込む
telco_url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
_req = urllib.request.Request(telco_url, headers={"User-Agent": "Mozilla/5.0"})
telco = pd.read_csv(io.BytesIO(urllib.request.urlopen(_req).read()))

# 前処理：使わないID列を消す／料金を数値に直す／予測対象を 0,1 にする
telco = telco.drop(columns=["customerID"])
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)
y_telco = (telco["Churn"] == "Yes").astype(int)   # 解約=1, 継続=0
telco = telco.drop(columns=["Churn"])

print(f"顧客の数: {len(telco)}人 / 情報の種類: {telco.shape[1]}個")
print(f"継続（No）: {(y_telco==0).sum()}人 / 解約（Yes）: {(y_telco==1).sum()}人")
display(telco.head(3))

列名（英語）の意味は、次のとおりです。

| 列名 | 意味 |
|---|---|
| gender | 性別（Female=女性 / Male=男性） |
| SeniorCitizen | 高齢者かどうか（1=高齢者 / 0=そうでない） |
| Partner | 配偶者・パートナーがいるか（Yes / No） |
| Dependents | 扶養家族がいるか（Yes / No） |
| tenure | 契約してからの月数 |
| PhoneService | 電話サービスの契約（Yes / No） |
| MultipleLines | 複数回線の契約（Yes / No / No phone service=電話契約なし） |
| InternetService | ネット回線の種類（DSL / Fiber optic=光回線 / No=契約なし） |
| OnlineSecurity | セキュリティオプション（Yes / No / No internet service=ネット契約なし） |
| OnlineBackup | バックアップオプション（Yes / No / ネット契約なし） |
| DeviceProtection | 端末保証オプション（Yes / No / ネット契約なし） |
| TechSupport | 技術サポート（Yes / No / ネット契約なし） |
| StreamingTV | TV配信サービス（Yes / No / ネット契約なし） |
| StreamingMovies | 映画配信サービス（Yes / No / ネット契約なし） |
| Contract | 契約期間（Month-to-month=月単位 / One year=1年 / Two year=2年） |
| PaperlessBilling | ペーパーレス請求（Yes / No） |
| PaymentMethod | 支払い方法（電子小切手 / 郵送小切手 / 銀行振込 / クレジットカード） |
| MonthlyCharges | 月額料金 |
| TotalCharges | これまでの合計支払額 |

予測対象は **Churn**（解約したか：Yes=解約 / No=継続）です。

### どのグループがどの情報を持つか（分割の指定）

3つのグループに含める情報は、次のように **最初から決まっています**。

| グループ | 含まれる情報 | 列名 |
|---|---|---|
| グループ1（顧客の属性） | 顧客の属性 | gender, SeniorCitizen, Partner, Dependents |
| グループ2（契約サービス） | 契約しているサービス | PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies |
| グループ3（契約と料金） | 契約期間と料金 | tenure, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges |

この指定に従って、次のセルの空欄を埋めてください。

In [ ]:
# 【穴埋め①】上の表の指定どおり、3つのグループが持つ列のリストを作る
dept_customer = ____
dept_service  = ____
dept_contract = ____

dept_cols = [dept_customer, dept_service, dept_contract]
dept_names = ["グループ1（顧客の属性）", "グループ2（契約サービス）", "グループ3（契約と料金）"]

# 【穴埋め②】各グループの列を、機械学習で使える数値の表に変換する（ワンホット化）
#   ヒント：pd.get_dummies(...) を使い、最後に .astype(np.float32) で数値の型をそろえる
def make_partition(cols):
    return ____

telco_partitions = [make_partition(cols) for cols in dept_cols]

# 各グループが何個の特徴を持つか確認
size_df = pd.DataFrame([
    {"グループ": name, "元の列数": len(cols), "変換後の特徴数": p.shape[1]}
    for name, cols, p in zip(dept_names, dept_cols, telco_partitions)])
display(size_df)

# ✅ 答え合わせ（穴埋め①②のチェック）
_ok = True
if sorted(dept_customer + dept_service + dept_contract) != sorted(telco.columns):
    print("❌ 穴埋め①：3グループの列を合わせると、全部の列がちょうど1回ずつ現れるはずです。指定を見直しましょう。")
    _ok = False
if not all(str(p.values.dtype) == "float32" for p in telco_partitions):
    print("❌ 穴埋め②：pd.get_dummies(...) のあと .astype(np.float32) になっているか確認しましょう。")
    _ok = False
if _ok:
    print("✅ OK：3つのグループに、過不足なく列を振り分けられています。")

In [ ]:
# 訓練用とテスト用に分ける（どの顧客を訓練/テストにするかを番号で決める）
telco_idx_tr, telco_idx_te = train_test_split(
    np.arange(len(telco)), test_size=0.2, random_state=42, stratify=y_telco)
y_telco_t = torch.tensor(y_telco.values, dtype=torch.float32).unsqueeze(1)
y_telco_tr, y_telco_te = y_telco_t[telco_idx_tr], y_telco_t[telco_idx_te]
print("データ準備 完了")

### 4-2. 垂直連合学習を自分で書く（穴埋め③〜⑩・難しめ）

ここが発展課題の本番です。例題の `run_vertical_fl` を見ながら、関数の中身を広い範囲まで自分で書いてください。
各空欄には、何をするかのコメントが付いています。骨組み（for ループや return）は残してあります。

In [ ]:
def run_vertical_fl_advanced(partitions, idx_tr, idx_te, ytr, yte,
                             rounds=200, client_lr=0.5, server_lr=0.01, seed=0):
    torch.manual_seed(seed)

    # 各グループ：自分のデータを標準化して持つ（訓練用の値で基準を作り、訓練用とテスト用の両方に適用）
    client_data_tr, client_data_te = [], []
    for p in partitions:
        arr = p.values
        # 【穴埋め③】訓練用データ arr[idx_tr] を使って標準化の基準（scaler）を作る
        scaler = ____
        # 【穴埋め④】その基準で訓練用・テスト用を変換し、torch.float32 のテンソルにして各リストに追加する
        client_data_tr.append(____)
        client_data_te.append(____)

    # 【穴埋め⑤】各グループのモデル（ClientModel）を、特徴数 d.shape[1] で作りリストにする
    client_models = ____
    # 【穴埋め⑥】各グループのオプティマイザ（SGD・学習率 client_lr）を作りリストにする
    client_opts   = ____
    # 【穴埋め⑦】サーバーモデル（ServerModel）を、入力サイズ EMB_DIM×グループ数 で作る
    server_model  = ____
    server_opt    = torch.optim.Adam(server_model.parameters(), lr=server_lr)
    criterion     = nn.BCELoss()

    history = []
    for r in range(rounds):
        # === ステップ① 各グループが中間の特徴を計算してサーバーへ送る ===
        # 【穴埋め⑧】各グループモデル m に自分のデータ d を入れて、中間の特徴のリストを作る
        embeddings = ____
        emb_cat = torch.cat([e.detach() for e in embeddings], dim=1).requires_grad_()

        # === ステップ② サーバーが中間の特徴を結合して学習する ===
        # 【穴埋め⑨】サーバーモデルに emb_cat を入れて予測 output を出し、正解 ytr との誤差 loss を計算する
        output = ____
        loss = ____
        server_opt.zero_grad(); loss.backward(); server_opt.step()

        # === ステップ③ 中間の特徴に対する勾配を各グループに返し、各自が学習する ===
        grads = emb_cat.grad.split([EMB_DIM] * len(partitions), dim=1)
        # 【穴埋め⑩】各グループで、勾配の初期化 -> 自分の中間の特徴に勾配gを逆伝播 -> パラメータ更新 を行う
        for m, o, e, g in zip(client_models, client_opts, embeddings, grads):
            ____

        with torch.no_grad():
            emb_te = torch.cat([m(d) for m, d in zip(client_models, client_data_te)], dim=1)
            acc = ((server_model(emb_te) > 0.5).float() == yte).float().mean().item() * 100
        history.append(acc)

    return history[-1], history

#### ✅ 答え合わせ（穴埋め③〜⑩のチェック）

小さく試運転して、正しく書けているか確認します。

In [ ]:
# 短い学習（20ラウンド）で試運転して確認する
_check_acc, _ = run_vertical_fl_advanced(
    telco_partitions, telco_idx_tr, telco_idx_te, y_telco_tr, y_telco_te, rounds=20)
_baseline = max(y_telco.iloc[telco_idx_te].mean(),
                1 - y_telco.iloc[telco_idx_te].mean()) * 100
_threshold = _baseline + 3

print(f"試運転の精度: {_check_acc:.1f}%（合格ライン: {_threshold:.1f}% 以上）")
if _check_acc >= _threshold:
    print("✅ OK：正しく書けています。次に進みましょう。")
else:
    print("❌ もう一度確認しましょう。例題の run_vertical_fl と1行ずつ見比べてください。")
    print("   ・標準化（scaler）やモデル・オプティマイザの用意はできていますか？")
    print("   ・① 中間の特徴の計算 / ② 予測と誤差 / ③ 勾配を返して各自が学習、はできていますか？")

### 4-3. 集約と垂直連合学習を比べる

In [ ]:
%%time
# 比較用：集約（全グループのデータを1か所に集めて学習）は例題の run_centralized をそのまま使える
telco_cent_acc, telco_cent_hist = run_centralized(
    telco_partitions, telco_idx_tr, telco_idx_te, y_telco_tr, y_telco_te)
telco_vfl_acc, telco_vfl_hist = run_vertical_fl_advanced(
    telco_partitions, telco_idx_tr, telco_idx_te, y_telco_tr, y_telco_te)
telco_naive = max(y_telco.iloc[telco_idx_te].mean(),
                  1 - y_telco.iloc[telco_idx_te].mean()) * 100

print(f"集約            : {telco_cent_acc:.1f}%")
print(f"垂直FL          : {telco_vfl_acc:.1f}%")
print(f"いつも同じ答え（答えとして多い方）にした場合: {telco_naive:.1f}%")

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(telco_vfl_hist)+1), telco_vfl_hist, label="垂直連合学習（持ち寄り）")
plt.plot(range(1, len(telco_cent_hist)+1), telco_cent_hist, color="orange",
         label=f"集約 最終={telco_cent_acc:.1f}%")
plt.axhline(telco_naive, color="gray", linestyle=":",
            label=f"いつも同じ答え（答えとして多い方）にした場合 = {telco_naive:.1f}%")
plt.xlabel("ラウンド数 / エポック数"); plt.ylabel("テスト精度（%）")
plt.title("発展課題：電話会社の解約予測の垂直連合学習")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()